In [1]:
import psutil
nthreads = psutil.cpu_count(logical=False)
# nthreads = 1

In [2]:
print(f"Num of threads using: ", nthreads)

Num of threads using:  10


In [3]:
import os
os.environ['OMP_NUM_THREADS'] = str(nthreads)
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [4]:
os.getpid()

56270

In [5]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer
import differential_operators

import numpy as np
import igl

import matplotlib
from matplotlib import pyplot as plt

In [6]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [7]:
from curved_linesearch import visualization

In [8]:
import newton_flow
import newton_flow_utils as nfu

In [9]:
from Benchmark import helper_funcs

In [10]:
import parallelism

# Newton Flow acts as Param

In [11]:
model = 'cow2Disc.off'

In [12]:
m = helper_funcs.read_mesh(f'../../../Models/TableOneModels/{model}')
print(f"Model: {model} Vertices: {m.numVertices()}")
print(f"Model: {model} Elements: {m.numElements()}")

Model: cow2Disc.off Vertices: 4540
Model: cow2Disc.off Elements: 8626


In [13]:
uv = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh(np.zeros((m.numVertices(),2)), m.elements())
m_2d.reembedElements(m.vertices())


m_init_2d = mesh.Mesh('ToysMesh/cow_init_2d.obj')
uv.setVars(m_init_2d.vertices().ravel())
nf = newton_flow.symmetric_dirichlet(m_2d, uv)

In [14]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [nf])

## nf, prob, and opt settings

In [15]:
nf.elementHessianShift = 1e-11
prob.hessianShift = 0
prob.useRelativeHessianShift = False

In [16]:
FIX_VARS = False
always_project = False

In [17]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.options.niter = 200

In [18]:
prob.energy()

1917.5504228797113

In [19]:
# brek

## RS 2D class Extrapolator 

In [20]:
import rotation_strain_extrapolation

In [21]:
linear_extrapolator = rotation_strain_extrapolation.LinearExtrapolator()

In [22]:
RS_nf_extrapolator = rotation_strain_extrapolation.RSNewtonFlowExtrapolator(m_2d)

In [23]:
Pade_extrapolator = extra_utils.PadeExtrapolator(opt, 19, constant_speed=False)

In [24]:
Taylor_extrapolator = extra_utils.TaylorExtrapolator(opt, 3, constant_speed=False)

In [25]:
extrapolator_list = [RS_nf_extrapolator]
hybrid_extrapolator = extra_utils.HybridExtrapolator(extrapolator_list)

In [26]:
hybrid_extrapolator.extrapolators

In [27]:
# brek

# Optimize

In [28]:
opt.options.niter = 6
benchmark.reset()
opt.optimize()
benchmark.report()

0	1917.55	95.3656	0.5	0	1
1	479.532	51.0399	0.5	0	1
2	120.126	27.9874	0.5	0	1
3	30.4842	15.7634	0.5	0	1
4	8.4525	9.02274	0.5	0	1
5	3.36985	5.27548	0.5	0	1
6	2.33809	3.0794	0.5	0	1
Newton iterations	0.0288784	1
    Newton iterate	0.0282157	6
        Backtracking	0.000729791	6
            NewtonMultiobjectiveProblem.customFeasibleStepLength	3.95899e-06	6
        Compute descent direction	0.0231109	6
            newton_step	0.0231033	6
                Catamari Numeric Factorize	0.00614466	12
                    BlockRightLooking<2>	0.00600934	12
                        Construct child-to-parent map	8.1e-05	1
                Catamari Symbolic Factorize	0.0113148	1
                    CatamariConverter	0.000245584	1
                        Entry Generation	9.8e-05	1
                        toSymmetryMode	0.000143416	1
                            InOrderBuilder constructor	8.9625e-05	1
                                allocate Ai, Ax	3.275e-05	1
                                columnSizeCalcu

In [29]:
# brek

In [30]:
line_search_method = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)

In [31]:
# benchmark.reset()
# vertices_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, line_search_method, max_iters=15, verbose=True)
# benchmark.report()# 

In [32]:
prob.energy()

2.338088539910006

In [33]:
brutal_line_search = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)

ternary_line_search = opt_utils.TernaryLinesearch()
golden_section_search = opt_utils.GoldenSectionSearch()

parabola_fit_search = opt_utils.ParabolaFitSearch()

In [34]:
max_alpha = 10

brutal_line_search.max_alpha = max_alpha

ternary_line_search.max_alpha = max_alpha
ternary_line_search.ternary_tol = 1

golden_section_search.max_alpha = max_alpha
golden_section_search.golden_tol = 1

parabola_fit_search.max_alpha = max_alpha
parabola_fit_search.use_extrapolation = True

In [35]:
# line_search_method = parabola_fit_search
# line_search_method = golden_section_search
line_search_method = brutal_line_search
parallelism.set_gradient_assembly_num_threads(5)

In [36]:
opt.options.gradTol = 2e-8
# nf.elementHessianShift = 1e-11

In [37]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, RS_nf_extrapolator, line_search_method, grad_tol=opt.options.gradTol, max_iters=20, 
                                             verbose=True, newton_step_tol=1, max_extraNewton_stop_counter=None)
# opt.optimize()
benchmark.report()

0 2.114756860135792 4.236512672635142 5.249839603685179 True 0.7000000000000001 0 3.9002405418771358
1 2.1006007162727247 7.587108954244603 2.3735210701314386 True 0.49 0 1.1723381033996692
2 2.0959681987570784 11.923693450512264 1.5551020594529112 True 0.46 1 0.7189769210982321
3 2.0945335250331576 14.938310776614026 1.111436906909107 True 0.25 2 0.2784069814960018
4 2.0936459534723744 16.89863466867514 0.956578721310572 True 0.19 3 0.18195071635814428
5 2.0929603151933382 18.295800411820114 0.8605878371766408 True 0.17 4 0.14641572780905135
6 2.092371778707691 19.151390693873353 0.7863564123038993 True 0.16 5 0.12589389075392254
7 2.091828693573207 19.59631163353482 0.7248739325424435 True 0.16 6 0.11603907227300114
8 2.091300200307949 19.72684780502581 0.6705832313987895 True 0.17 7 0.11405080857946885
9 2.0907535717638583 19.67073273135266 0.6197874098059886 True 0.2 8 0.1240140616093019
10 2.0901513728815617 19.093297846389987 0.5679775814899135 True 0.24 9 0.13637761775575263
11 

In [ ]:
# benchmark.pieChart('call_linesearch_func:linesearch_eval:linesearch_eval_call_in_Cpp:getUVnewSolvePoisson:Poisson RHS extraction', includeOutside=False)
benchmark.pieChart('newton_extrapolate')

In [ ]:
benchmark.totalTime('Poisson Solver$')

In [ ]:
benchmark.totalTime('NewtonMultiobjectiveProblem.gradient$')

In [ ]:
benchmark.totalTime('Poisson RHS extraction$')

In [ ]:
prob.energy()

In [ ]:
np.linalg.norm(prob.gradient())

In [ ]:
vertices_list[0]

In [ ]:
prob.getVars()